In [1]:
!pip install spacy pandas pdfplumber
!python -m spacy download en_core_web_sm

   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   --- ------------------------------------ 1.3/14.2 MB 6.7 MB/s eta 0:00:02
   ------ --------------------------------- 2.4/14.2 MB 6.0 MB/s eta 0:00:02
   ---------- ----------------------------- 3.7/14.2 MB 5.9 MB/s eta 0:00:02
   ------------- -------------------------- 4.7/14.2 MB 5.9 MB/s eta 0:00:02
   ----------------- ---------------------- 6.3/14.2 MB 6.1 MB/s eta 0:00:02
   -------------------- ------------------- 7.3/14.2 MB 5.9 MB/s eta 0:00:02
   ------------------------- -------------- 8.9/14.2 MB 6.1 MB/s eta 0:00:01
   ---------------------------- ----------- 10.2/14.2 MB 6.2 MB/s eta 0:00:01
   -------------------------------- ------- 11.5/14.2 MB 6.1 MB/s eta 0:00:01
   ---------------------------------- ----- 12.3/14.2 MB 6.0 MB/s eta 0:00:01
   ------------------------------------ --- 13.1/14.2 MB 5.8 MB/s eta 0:00:01
   -------------------------------------- - 13.6/14.2 MB 5.6 MB/s eta 0:00:01
 

In [2]:
import os

for root, dirs, files in os.walk("data"):
    level = root.replace("data", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for f in files[:5]:  # just first 5 files per folder to keep it readable
        print(f"{subindent}{f}")

data/
  data/
    data/
      ACCOUNTANT/
        10554236.pdf
        10674770.pdf
        11163645.pdf
        11759079.pdf
        12065211.pdf
      ADVOCATE/
        10186968.pdf
        10344379.pdf
        10659182.pdf
        10818478.pdf
        11174187.pdf
      AGRICULTURE/
        10953078.pdf
        11197262.pdf
        11676151.pdf
        11813872.pdf
        12341902.pdf
      APPAREL/
        10182582.pdf
        10562768.pdf
        10738095.pdf
        10876132.pdf
        11232471.pdf
      ARTS/
        10830646.pdf
        11187796.pdf
        11360471.pdf
        11555549.pdf
        11995013.pdf
      AUTOMOBILE/
        11152490.pdf
        11257723.pdf
        11797122.pdf
        11887930.pdf
        14455622.pdf
      AVIATION/
        10176815.pdf
        10189110.pdf
        10395944.pdf
        10567764.pdf
        10945968.pdf
      BANKING/
        10329506.pdf
        10909673.pdf
        11065180.pdf
        11262933.pdf
        11266906.pdf
      B

In [3]:
import pandas as pd

df = pd.read_csv("data/Resume/Resume.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(2484, 4)
['ID', 'Resume_str', 'Resume_html', 'Category']


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
print(df['Resume_str'][0])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [5]:
import re

SECTION_HEADERS = ["Summary", "Highlights", "Accomplishments", "Experience",
                    "Education", "Skills", "Objective", "Certifications", "Interests"]

def split_into_sections(text):
    pattern = r'\b(' + '|'.join(SECTION_HEADERS) + r')\b'
    matches = list(re.finditer(pattern, text))

    sections = {}
    for i, match in enumerate(matches):
        header = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[header] = text[start:end].strip()
    return sections

def extract_skills(text):
    sections = split_into_sections(text)
    skills_text = sections.get("Skills", "")
    if not skills_text:
        return []
    return [s.strip() for s in skills_text.split(",") if s.strip()]

# Test on the sample resume
sample_text = df['Resume_str'][0]
sections = split_into_sections(sample_text)

print("Sections found:", list(sections.keys()))
print()
print("Skills extracted:")
print(extract_skills(sample_text))

Sections found: ['Summary', 'Highlights', 'Accomplishments', 'Experience', 'Education', 'Skills']

Skills extracted:
['Accounting', 'ads', 'advertising', 'analytical skills', 'benefits', 'billing', 'budgeting', 'clients', 'Customer Service', 'data analysis', 'delivery', 'documentation', 'employee relations', 'financial management', 'government relations', 'Human Resources', 'insurance', 'labor relations', 'layout', 'Marketing', 'marketing collateral', 'medical billing', 'medical terminology', 'office', 'organizational', 'payroll', 'performance reviews', 'personnel', 'policies', 'posters', 'presentations', 'public relations', 'purchasing', 'reporting', 'statistics', 'website.']


In [6]:
DEGREE_KEYWORDS = ["High School Diploma", "Associate", "Bachelor", "B.S.", "B.A.", "B.Tech",
                    "Master", "M.S.", "M.A.", "M.Tech", "MBA", "Ph.D.", "Doctorate",
                    "Diploma", "Certificate"]

INSTITUTION_KEYWORDS = ["University", "College", "Institute", "School", "Academy"]

def extract_education(text):
    sections = split_into_sections(text)
    edu_text = sections.get("Education", "")
    if not edu_text:
        return {"degrees": [], "institutions": [], "years": []}

    years = re.findall(r'\b(?:19|20)\d{2}\b', edu_text)

    degrees = [kw for kw in DEGREE_KEYWORDS if kw.lower() in edu_text.lower()]

    institutions = []
    for chunk in re.split(r'\s{2,}', edu_text):
        chunk = chunk.strip()
        if chunk and any(kw.lower() in chunk.lower() for kw in INSTITUTION_KEYWORDS):
            institutions.append(chunk)

    return {"degrees": degrees, "institutions": institutions, "years": years}

# Test on the sample resume
print(extract_education(sample_text))

{'degrees': ['High School Diploma', 'Diploma'], 'institutions': ['Jefferson College', 'High School Diploma', 'College Prep. studies', 'Awarded American Shrubel Leadership Scholarship to Jefferson College'], 'years': ['1999', '1998']}


In [7]:
def extract_education(text):
    sections = split_into_sections(text)
    edu_text = sections.get("Education", "")
    if not edu_text:
        return {"degrees": [], "institutions": [], "years": []}

    years = re.findall(r'\b(?:19|20)\d{2}\b', edu_text)

    # Match longest degree keywords first, skip if already covered by a longer match
    degrees = []
    for kw in sorted(DEGREE_KEYWORDS, key=len, reverse=True):
        if kw.lower() in edu_text.lower() and not any(kw.lower() in d.lower() for d in degrees):
            degrees.append(kw)

    # Institution names sit right before " － City" in this dataset's format
    institutions = re.findall(r"([A-Za-z&,.' ]+?)\s+－\s+City", edu_text)
    institutions = [i.strip() for i in institutions]

    return {"degrees": degrees, "institutions": institutions, "years": years}

# Re-test
print(extract_education(sample_text))

{'degrees': ['High School Diploma'], 'institutions': ['Jefferson College', 'Sainte Genevieve Senior High'], 'years': ['1999', '1998']}


In [8]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_name(text):
    doc = nlp(text[:300])  # names usually appear near the top
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    return None

def extract_email(text):
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else None

def extract_phone(text):
    match = re.search(r'(\+?\d{1,2}[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}', text)
    return match.group(0) if match else None

# Test 1: on the anonymized dataset sample (expect None / None / None)
print("--- Dataset sample ---")
print("Name:", extract_name(sample_text))
print("Email:", extract_email(sample_text))
print("Phone:", extract_phone(sample_text))

# Test 2: on a synthetic resume snippet (expect real values)
test_text = """
John Smith
Email: john.smith@example.com
Phone: (555) 123-4567

Summary
Experienced software engineer with 5 years in backend development.
"""

print("\n--- Synthetic test ---")
print("Name:", extract_name(test_text))
print("Email:", extract_email(test_text))
print("Phone:", extract_phone(test_text))

--- Dataset sample ---
Name: None
Email: None
Phone: None

--- Synthetic test ---
Name: John Smith
Email: john.smith@example.com
Phone: (555) 123-4567


In [9]:
def parse_resume(text):
    return {
        "name": extract_name(text),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "skills": extract_skills(text),
        "education": extract_education(text)
    }

# Quick sanity check on the sample
import pprint
pprint.pprint(parse_resume(sample_text))

{'education': {'degrees': ['High School Diploma'],
               'institutions': ['Jefferson College',
                                'Sainte Genevieve Senior High'],
               'years': ['1999', '1998']},
 'email': None,
 'name': None,
 'phone': None,
 'skills': ['Accounting',
            'ads',
            'advertising',
            'analytical skills',
            'benefits',
            'billing',
            'budgeting',
            'clients',
            'Customer Service',
            'data analysis',
            'delivery',
            'documentation',
            'employee relations',
            'financial management',
            'government relations',
            'Human Resources',
            'insurance',
            'labor relations',
            'layout',
            'Marketing',
            'marketing collateral',
            'medical billing',
            'medical terminology',
            'office',
            'organizational',
            'payroll',
          

In [10]:
df['Skills_List'] = df['Resume_str'].apply(extract_skills)
df['Skill_Count'] = df['Skills_List'].apply(len)

print("Average skills listed per resume, by category:")
print(df.groupby('Category')['Skill_Count'].mean().sort_values(ascending=False))

from collections import Counter
all_skills = [skill for skills in df['Skills_List'] for skill in skills]
top_skills = Counter(all_skills).most_common(15)
print("\nTop 15 most common skills across all resumes:")
for skill, count in top_skills:
    print(f"{skill}: {count}")

Average skills listed per resume, by category:
Category
INFORMATION-TECHNOLOGY    44.100000
CONSULTANT                35.817391
BPO                       34.636364
ENGINEERING               33.203390
ACCOUNTANT                31.542373
FINANCE                   30.779661
BANKING                   29.095652
DIGITAL-MEDIA             27.666667
HR                        27.609091
PUBLIC-RELATIONS          27.252252
ADVOCATE                  24.711864
BUSINESS-DEVELOPMENT      24.550000
DESIGNER                  23.401869
HEALTHCARE                23.260870
AUTOMOBILE                22.388889
AVIATION                  22.341880
APPAREL                   21.257732
CONSTRUCTION              20.741071
AGRICULTURE               17.650794
ARTS                      17.252427
FITNESS                   15.752137
SALES                     14.431034
CHEF                      14.288136
TEACHER                   13.921569
Name: Skill_Count, dtype: float64

Top 15 most common skills across all resumes:

In [11]:
app_code = '''
import spacy
import re
import pdfplumber
import streamlit as st

nlp = spacy.load("en_core_web_sm")

SECTION_HEADERS = ["Summary", "Highlights", "Accomplishments", "Experience",
                    "Education", "Skills", "Objective", "Certifications", "Interests"]

DEGREE_KEYWORDS = ["High School Diploma", "Associate", "Bachelor", "B.S.", "B.A.", "B.Tech",
                    "Master", "M.S.", "M.A.", "M.Tech", "MBA", "Ph.D.", "Doctorate",
                    "Diploma", "Certificate"]

INSTITUTION_KEYWORDS = ["University", "College", "Institute", "School", "Academy"]

def split_into_sections(text):
    pattern = r"\\b(" + "|".join(SECTION_HEADERS) + r")\\b"
    matches = list(re.finditer(pattern, text))
    sections = {}
    for i, match in enumerate(matches):
        header = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[header] = text[start:end].strip()
    return sections

def extract_skills(text):
    sections = split_into_sections(text)
    skills_text = sections.get("Skills", "")
    if not skills_text:
        return []
    return [s.strip().rstrip(".") for s in skills_text.split(",") if s.strip()]

def extract_education(text):
    sections = split_into_sections(text)
    edu_text = sections.get("Education", "")
    if not edu_text:
        return {"degrees": [], "institutions": [], "years": []}
    years = re.findall(r"\\b(?:19|20)\\d{2}\\b", edu_text)
    degrees = []
    for kw in sorted(DEGREE_KEYWORDS, key=len, reverse=True):
        if kw.lower() in edu_text.lower() and not any(kw.lower() in d.lower() for d in degrees):
            degrees.append(kw)
    institutions = re.findall(r"([A-Za-z&,.\\' ]+?)\\s+[\\u2010-\\u2015-]\\s+City", edu_text)
    institutions = [i.strip() for i in institutions]
    return {"degrees": degrees, "institutions": institutions, "years": years}

def extract_name(text):
    doc = nlp(text[:300])
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    return None

def extract_email(text):
    match = re.search(r"[\\w.-]+@[\\w.-]+\\.\\w+", text)
    return match.group(0) if match else None

def extract_phone(text):
    match = re.search(r"(\\+?\\d{1,2}[\\s.-]?)?\\(?\\d{3}\\)?[\\s.-]?\\d{3}[\\s.-]?\\d{4}", text)
    return match.group(0) if match else None

def parse_resume(text):
    return {
        "name": extract_name(text),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "skills": extract_skills(text),
        "education": extract_education(text)
    }

def extract_text_from_pdf(uploaded_file):
    text = ""
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text

st.set_page_config(page_title="AI Resume Parser", page_icon="📄", layout="centered")
st.title("📄 AI Resume Parser")
st.write("Upload a resume (PDF) and get structured information extracted automatically.")

uploaded_file = st.file_uploader("Upload a resume PDF", type=["pdf"])

if uploaded_file is not None:
    with st.spinner("Extracting text from PDF..."):
        text = extract_text_from_pdf(uploaded_file)

    if not text.strip():
        st.error("Could not extract text from this PDF. It may be a scanned image rather than text-based.")
    else:
        with st.spinner("Parsing resume..."):
            result = parse_resume(text)

        st.success("Resume parsed successfully!")

        col1, col2 = st.columns(2)
        with col1:
            st.subheader("👤 Contact Info")
            st.write(f"**Name:** {result[\\'name\\'] or \\'Not found\\'}")
            st.write(f"**Email:** {result[\\'email\\'] or \\'Not found\\'}")
            st.write(f"**Phone:** {result[\\'phone\\'] or \\'Not found\\'}")

        with col2:
            st.subheader("🎓 Education")
            edu = result["education"]
            if edu["degrees"]:
                st.write(f"**Degree(s):** {\\', \\'.join(edu[\\'degrees\\'])}")
            if edu["institutions"]:
                st.write(f"**Institution(s):** {\\', \\'.join(edu[\\'institutions\\'])}")
            if edu["years"]:
                st.write(f"**Year(s):** {\\', \\'.join(edu[\\'years\\'])}")
            if not edu["degrees"] and not edu["institutions"]:
                st.write("Not found")

        st.subheader("🛠️ Skills")
        if result["skills"]:
            st.write(", ".join(result["skills"]))
        else:
            st.write("Not found")

        with st.expander("View raw extracted text"):
            st.text(text)
else:
    st.info("👆 Upload a PDF resume to get started.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [12]:
%%writefile app.py
import spacy
import re
import pdfplumber
import streamlit as st

nlp = spacy.load("en_core_web_sm")

SECTION_HEADERS = ["Summary", "Highlights", "Accomplishments", "Experience",
                    "Education", "Skills", "Objective", "Certifications", "Interests"]

DEGREE_KEYWORDS = ["High School Diploma", "Associate", "Bachelor", "B.S.", "B.A.", "B.Tech",
                    "Master", "M.S.", "M.A.", "M.Tech", "MBA", "Ph.D.", "Doctorate",
                    "Diploma", "Certificate"]

INSTITUTION_KEYWORDS = ["University", "College", "Institute", "School", "Academy"]

def split_into_sections(text):
    pattern = r'\b(' + '|'.join(SECTION_HEADERS) + r')\b'
    matches = list(re.finditer(pattern, text))
    sections = {}
    for i, match in enumerate(matches):
        header = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[header] = text[start:end].strip()
    return sections

def extract_skills(text):
    sections = split_into_sections(text)
    skills_text = sections.get("Skills", "")
    if not skills_text:
        return []
    return [s.strip().rstrip('.') for s in skills_text.split(",") if s.strip()]

def extract_education(text):
    sections = split_into_sections(text)
    edu_text = sections.get("Education", "")
    if not edu_text:
        return {"degrees": [], "institutions": [], "years": []}
    years = re.findall(r'\b(?:19|20)\d{2}\b', edu_text)
    degrees = []
    for kw in sorted(DEGREE_KEYWORDS, key=len, reverse=True):
        if kw.lower() in edu_text.lower() and not any(kw.lower() in d.lower() for d in degrees):
            degrees.append(kw)
    institutions = re.findall(r"([A-Za-z&,.' ]+?)\s+[－-]\s+City", edu_text)
    institutions = [i.strip() for i in institutions]
    return {"degrees": degrees, "institutions": institutions, "years": years}

def extract_name(text):
    doc = nlp(text[:300])
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    return None

def extract_email(text):
    match = re.search(r'[\w.-]+@[\w.-]+\.\w+', text)
    return match.group(0) if match else None

def extract_phone(text):
    match = re.search(r'(\+?\d{1,2}[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}', text)
    return match.group(0) if match else None

def parse_resume(text):
    return {
        "name": extract_name(text),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "skills": extract_skills(text),
        "education": extract_education(text)
    }

def extract_text_from_pdf(uploaded_file):
    text = ""
    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text

st.set_page_config(page_title="AI Resume Parser", page_icon="📄", layout="centered")
st.title("📄 AI Resume Parser")
st.write("Upload a resume (PDF) and get structured information extracted automatically.")

uploaded_file = st.file_uploader("Upload a resume PDF", type=["pdf"])

if uploaded_file is not None:
    with st.spinner("Extracting text from PDF..."):
        text = extract_text_from_pdf(uploaded_file)

    if not text.strip():
        st.error("Could not extract text from this PDF. It may be a scanned image rather than text-based.")
    else:
        with st.spinner("Parsing resume..."):
            result = parse_resume(text)

        st.success("Resume parsed successfully!")

        col1, col2 = st.columns(2)
        with col1:
            st.subheader("👤 Contact Info")
            name = result['name'] or 'Not found'
            email = result['email'] or 'Not found'
            phone = result['phone'] or 'Not found'
            st.write(f"**Name:** {name}")
            st.write(f"**Email:** {email}")
            st.write(f"**Phone:** {phone}")

        with col2:
            st.subheader("🎓 Education")
            edu = result['education']
            if edu['degrees']:
                st.write(f"**Degree(s):** {', '.join(edu['degrees'])}")
            if edu['institutions']:
                st.write(f"**Institution(s):** {', '.join(edu['institutions'])}")
            if edu['years']:
                st.write(f"**Year(s):** {', '.join(edu['years'])}")
            if not edu['degrees'] and not edu['institutions']:
                st.write("Not found")

        st.subheader("🛠️ Skills")
        if result['skills']:
            st.write(", ".join(result['skills']))
        else:
            st.write("Not found")

        with st.expander("View raw extracted text"):
            st.text(text)
else:
    st.info("👆 Upload a PDF resume to get started.")

Overwriting app.py


In [13]:
readme_content = """# AI Resume Parser

An AI-powered resume parsing tool built for the Pinnacle Labs internship program. Upload a resume (PDF) and automatically extract structured information: name, email, phone, skills, and education.

## Features
- Upload any PDF resume and get instant structured extraction
- Uses spaCy NLP (Named Entity Recognition) to detect candidate names
- Regex-based extraction for email addresses and phone numbers
- Section-aware parsing for Skills and Education using rule-based text segmentation
- Built and validated on the [Resume Dataset (Kaggle)](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset) — 2,484 resumes across 24 job categories
- Interactive Streamlit web app

## Tech Stack
- Python
- spaCy (NLP / Named Entity Recognition)
- pdfplumber (PDF text extraction)
- pandas (data handling and EDA)
- Streamlit (web app)

## Dataset Insights (from EDA)
- IT resumes listed the most skills on average (~44 per resume)
- Most common skills across all resumes: Excel, client communication, sales, quality, meetings
- Education and Skills sections are reliably structured; contact info varies by resume source

## How to Run
1. Clone this repository
2. Install dependencies:
\`\`\`
pip install -r requirements.txt
python -m spacy download en_core_web_sm
\`\`\`
3. Run the app:
\`\`\`
streamlit run app.py
\`\`\`
4. Upload a PDF resume and view the parsed results

## Project Structure
\`\`\`
PinnacleLabs_ResumeParser/
├── app.py                  # Streamlit app
├── resume_parser.ipynb     # Development notebook (EDA + extraction logic)
├── requirements.txt
└── README.md
\`\`\`

## About
Built as part of the Pinnacle Labs 2026 Internship Program.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

requirements_content = """spacy
pandas
pdfplumber
streamlit
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

print("README.md and requirements.txt created successfully!")

README.md and requirements.txt created successfully!


<>:28: SyntaxWarning: invalid escape sequence '\`'
<>:28: SyntaxWarning: invalid escape sequence '\`'
C:\Users\ashwi\AppData\Local\Temp\ipykernel_14132\1971602710.py:28: SyntaxWarning: invalid escape sequence '\`'
  \`\`\`


In [14]:
%%writefile README.md
# AI Resume Parser

An AI-powered resume parsing tool built for the Pinnacle Labs internship program. Upload a resume (PDF) and automatically extract structured information: name, email, phone, skills, and education.

## Features
- Upload any PDF resume and get instant structured extraction
- Uses spaCy NLP (Named Entity Recognition) to detect candidate names
- Regex-based extraction for email addresses and phone numbers
- Section-aware parsing for Skills and Education using rule-based text segmentation
- Built and validated on the [Resume Dataset (Kaggle)](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset) — 2,484 resumes across 24 job categories
- Interactive Streamlit web app

## Tech Stack
- Python
- spaCy (NLP / Named Entity Recognition)
- pdfplumber (PDF text extraction)
- pandas (data handling and EDA)
- Streamlit (web app)

## Dataset Insights (from EDA)
- IT resumes listed the most skills on average (~44 per resume)
- Most common skills across all resumes: Excel, client communication, sales, quality, meetings
- Education and Skills sections are reliably structured; contact info varies by resume source

## How to Run
1. Clone this repository
2. Install dependencies:

Overwriting README.md


In [15]:
%%writefile .gitignore
data/
__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store

Writing .gitignore
